# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The FAIR^2 dataset is provided via the following Croissant schema URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Croissant schema URL for FAIR^2 data
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset (schema and all metadata)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All record sets and fields are referenced by their `@id`s, per Croissant specification.

In [ ]:
# List record sets with their @id's
record_sets = dataset.record_sets()
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For demonstration, let's examine the fields and columns of each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs.get('name', 'N/A')} (@id: {rs['@id']})")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields (@id):")
        for field in fields:
            if isinstance(field, dict):
                f_id = field.get('@id', str(field))
                f_name = field.get('name', 'N/A')
            else:
                f_id = str(field)
                f_name = 'N/A'
            print(f"    - {f_id} (name: {f_name})")
    if 'column' in rs:
        cols = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        print("  Columns (@id):")
        for col in cols:
            if isinstance(col, dict):
                c_id = col.get('@id', str(col))
                c_name = col.get('name', 'N/A')
            else:
                c_id = str(col)
                c_name = 'N/A'
            print(f"    - {c_id} (name: {c_name})")

## 3. Data Extraction
Load data from specific record sets into DataFrames for further exploration. 

Below, we'll dynamically load all available record sets using their `@id`. You may customize which record sets to load by their `@id`.

In [ ]:
dataframes = {}
rs_ids = [rs['@id'] for rs in dataset.record_sets()]

for record_set_id in rs_ids:
    print(f"\nExtracting records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns:")
        print(f"    {df.columns.tolist()}")
    else:
        print("  No records found.")

# For exploration, let's pick the first record set with data
primary_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        primary_record_set_id = k
        break

if primary_record_set_id:
    print(f"\nSample data from record set @id: {primary_record_set_id}")
    display(dataframes[primary_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple data processing steps on the chosen primary record set (referenced by its `@id`).

We will:
- Select a numeric field (`@id`) for filtering and normalization.
- Filter records where the selected numeric field exceeds a threshold.
- Normalize that field.
- Optionally group by a categorical field (by its `@id`).


In [ ]:
import numpy as np

if primary_record_set_id and not dataframes[primary_record_set_id].empty:
    df = dataframes[primary_record_set_id]
    # Attempt to automatically detect numeric fields (@id)
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_fields:
        # Try to coerce all columns to numeric to find candidates
        candidate_fields = []
        for col in df.columns:
            try:
                temp = pd.to_numeric(df[col], errors='coerce')
                if temp.notnull().sum() > 0:
                    candidate_fields.append(col)
            except:
                continue
        numeric_fields = candidate_fields
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Analyzing numeric field @id: {numeric_field_id}")
        # Convert to numeric if needed
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a likely categorical field
        candidate_group_fields = [col for col in df.columns if df[col].nunique() < len(df) // 2 and not np.issubdtype(df[col].dtype, np.number)]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric fields detected in the selected record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the distribution of the key numeric field and, if possible, compare groups by category (using `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and not dataframes[primary_record_set_id].empty and 'numeric_field_id' in locals():
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    ax.set_xlabel(numeric_field_id)
    plt.show()
    # If grouping field was found, draw a boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('Visualization is not possible: no numeric field detected or data missing.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 dataset using `mlcroissant`. We inspected available record sets (referenced by their `@id`), examined fields, extracted and processed data, and visualized numeric distributions. For further analysis, use the column and field `@id` references discovered in Section 2 to ensure reproducibility and clarity across code and documentation.
